# NorESM CCN–CDNC Susceptibility Builder

**Scientific question:** How sensitive is cloud droplet number concentration (CDNC) to cloud condensation nuclei (CCN) across stations, vertical levels, and CCN cutoff radii — and how much does the answer depend on the regression estimator used (OLS vs total-least-squares / Deming / PCA)?

Susceptibility is the slope d ln(CDNC) / d ln(CCN). For each station it is computed two ways: **per vertical level** (slope vs lev, per radius) and **all-levels pooled** (one slope per radius, levels+time flattened). Four estimators are run side-by-side because OLS is biased low when the x-variable (CCN) carries measurement/sampling noise, so the error-in-variables fits (TLS/Deming/PCA) bracket the true slope.

**Input:** per-station NorESM/OsloAero file on `/share/sabl0586/...`  
**Output:** `Susceptibility_all` (per-level + all-level slopes/intercepts, all stations) and `CCN_all` (aligned CCN/CDNC for plotting).

---
*Notebook structure (collapse the `##` headers to skim): 1 Setup · 2 Fit functions · 3 Load · 4 Build · 5 Plots.*

## 1 · Setup, imports and configuration

All imports and the single source-of-truth for paths and parameters live here. `PeterChurchillFunctions` (imported as `Function`) provides the estimators (`OLS_fit`, `TLS_fit`, `deming_fit`, `PCA_fit`), the NorESM extractor `NorESMExtract_Dask`, the CCN integrator `NorERF`, and the plotting helper `plot_hexbin_regression_multi`.

In [ ]:
# --- Imports ---
import PeterChurchillFunctions as Function
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf
import statsmodels.api as sm
from scipy.odr import ODR, Model, RealData
from matplotlib.colors import LogNorm

# --- Configuration (single source of truth) ---
# Per-station NorESM/OsloAero output (15 stations; dims: station, radius, time, lev).
NOR_PATH = "/share/sabl0586/all_stations_NorESM_OsloAero_prcp2szdst_f19_f19_noresmv211_corr_ilevall_levs_4Peter.nc"

# CCN cutoff radii to evaluate susceptibility at, in nm (20..50 inclusive).
RADII = np.arange(20, 51)

# Diameter grid for the size-distribution integration inside NorERF (log-spaced, nm).
D_GRID = xr.DataArray(
    np.logspace(-0.5, 3, num=200),
    dims=['D'],
    coords={'D': np.logspace(-0.5, 3, num=200)},
)

# Estimator labels used throughout. NOTE: the per-level builder labels the
# total-least-squares fit 'ODR' while the all-level builder labels the SAME fit
# 'TLS' — they are the same method (scipy.odr under the hood), just named
# differently in the two output Datasets. Kept as-is to avoid breaking downstream.
FIT_TYPES = ['OLS', 'ODR', 'Deming', 'PCA']        # per-level var names: '<FIT> slope'
ALL_LEVEL_METHODS = ['OLS', 'TLS', 'Deming', 'PCA']  # all-level var names: 'All_Level_<M>_slope'

## 2 · Susceptibility fit functions

Two builders, both fitting in **log10–log10 space** (so the slope is the susceptibility, a power-law exponent). Each wraps four estimators via `xr.apply_ufunc` with `vectorize=True` and `dask='parallelized'` so the fit runs per grid-cell over the chosen core dimensions.

- `Susceptibility_by_Level` — core dim `time`; returns a slope **per (radius, lev)** plus the aligned CCN/CDNC used.
- `compute_allLev` — core dims `lev`+`time` pooled; returns one slope **per radius**. CDNC is broadcast up to CCN's shape first so the two share `(radius, lev, time)`.

In [ ]:
def Susceptibility_by_Level(CCN_ds, CDNC_da):
    """
    Compute OLS slope/intercept between CCN(radius, lev, time)
    and CDNC(lev, time) across 'time'.
    """
    CCN_aligned, CDNC_aligned = xr.align(CCN_ds, CDNC_da)

    # Fit in log10 space, reducing over 'time' -> one slope per (radius, lev).
    OLS_slope, OLS_intercept = xr.apply_ufunc(
        Function.OLS_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_aligned),
        input_core_dims=[['time'], ['time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )
    # Total least squares (a.k.a. ODR here): accounts for noise in CCN as well as CDNC.
    ODR_slope, ODR_intercept = xr.apply_ufunc(
        Function.TLS_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_aligned),
        input_core_dims=[['time'], ['time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )
    Deming_slope, Deming_intercept = xr.apply_ufunc(
        Function.deming_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_aligned),
        input_core_dims=[['time'], ['time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )
    PCA_slope, PCA_intercept = xr.apply_ufunc(
        Function.PCA_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_aligned),
        input_core_dims=[['time'], ['time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )

    ds_out = xr.Dataset(
        data_vars={
            'OLS slope': (('radius', 'lev'), OLS_slope.data),
            'OLS intercept': (('radius', 'lev'), OLS_intercept.data),
            'ODR slope': (('radius', 'lev'), ODR_slope.data),
            'ODR intercept': (('radius', 'lev'), ODR_intercept.data),
            'Deming slope': (('radius', 'lev'), Deming_slope.data),
            'Deming intercept': (('radius', 'lev'), Deming_intercept.data),
            'PCA slope': (('radius', 'lev'), PCA_slope.data),
            'PCA intercept': (('radius', 'lev'), PCA_intercept.data),

        },
        coords={
            'radius': CCN_aligned.radius,
            'lev': CCN_aligned.lev,

        }
    )
    ds_out2 = xr.Dataset(
        data_vars={
            'CCN': CCN_aligned,       # dims: (radius, lev, time)
            'CDNC': CDNC_aligned      # dims: (lev, time)
        },
        coords={
            'radius': CCN_aligned.radius,
            'lev': CCN_aligned.lev,
            'time': CCN_aligned.time
        }
    )

    return ds_out, ds_out2


In [ ]:
def compute_allLev(CCN_ds, CDNC_da):
    """
    Compute OLS and TLS slope/intercept per radius,
    flattening across all levels and times.
    Returns one slope+intercept per radius for all four estimators,
    as variables 'All_Level_<METHOD>_slope' / '..._intercept'
    with METHOD in {OLS, TLS, Deming, PCA}.
    """
    # Align CCN and CDNC over lev and time
    CCN_aligned, CDNC_aligned = xr.align(CCN_ds, CDNC_da)

    # Broadcast CDNC to match CCN dimensions
    CDNC_broadcast = CDNC_aligned.broadcast_like(CCN_aligned)

    # --- OLS ---
    slope_OLS, intercept_OLS = xr.apply_ufunc(
        Function.OLS_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_broadcast),
        input_core_dims=[['lev', 'time'], ['lev', 'time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )

    # --- TLS ---
    slope_TLS, intercept_TLS = xr.apply_ufunc(
        Function.TLS_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_broadcast),
        input_core_dims=[['lev', 'time'], ['lev', 'time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )
    
    # --- Deming ---
    slope_Deming, intercept_Deming = xr.apply_ufunc(
        Function.deming_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_broadcast),
        input_core_dims=[['lev', 'time'], ['lev', 'time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )
    # --- PCA ---
    slope_PCA, intercept_PCA = xr.apply_ufunc(
        Function.PCA_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_broadcast),
        input_core_dims=[['lev', 'time'], ['lev', 'time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )


    # Package into dataset
    ds_out = xr.Dataset(
        data_vars={
            'All_Level_OLS_slope': (('radius',), slope_OLS.data),
            'All_Level_OLS_intercept': (('radius',), intercept_OLS.data),
            'All_Level_TLS_slope': (('radius',), slope_TLS.data),
            'All_Level_TLS_intercept': (('radius',), intercept_TLS.data),
            'All_Level_Deming_slope': (('radius',), slope_Deming.data),
            'All_Level_Deming_intercept': (('radius',), intercept_Deming.data),
            'All_Level_PCA_slope': (('radius',), slope_PCA.data),
            'All_Level_PCA_intercept': (('radius',), intercept_PCA.data),

        },
        coords={
            'radius': CCN_aligned.radius,

        }
    )

    return ds_out

## 3 · Load the NorESM dataset

Open lazily (`chunks={}` keeps it dask-backed) and pull the station list. `VarList` empty means the extractor uses its default variable set.

In [ ]:
ds = xr.open_dataset(NOR_PATH, chunks={})
stations = ds["station"].values
VarList = []   # empty -> NorESMExtract_Dask uses its defaults

## 4 · Build susceptibility across all stations

For each station: extract variables, integrate the size distribution into CCN at each cutoff radius (`NorERF`), then run both builders and merge. Results are tagged with a `station` coord and concatenated. The two `.compute()` calls force the lazy dask graph so everything downstream is in-memory.

*(This is the expensive cell — collapse it once it has run.)*

In [ ]:
all_stations = []
all_stationsCCN = []

for station in stations:
    # Extract this station's variables, then integrate aerosol size dist -> CCN(radius).
    Nor_ds = Function.NorESMExtract_Dask(NOR_PATH, station, VarList, D_GRID, PNSD=False)
    CCN_ds = Function.NorERF(Nor_ds, RADII)

    # Per-level and all-level susceptibility for this station.
    Level_susc_ds, Levels_CCN_ds = Susceptibility_by_Level(CCN_ds, Nor_ds['CDNC'])
    Susceptibility_AllLevs = compute_allLev(CCN_ds, Nor_ds['CDNC'])

    Susceptibility_ds = xr.merge([Level_susc_ds, Susceptibility_AllLevs])

    all_stations.append(Susceptibility_ds.assign_coords(station=station))
    all_stationsCCN.append(Levels_CCN_ds.assign_coords(station=station))

# Concatenate across stations.
# NOTE: Nor_ds / CCN_ds persist as the LAST station's values after the loop;
# the per-station plot in section 5 relies on that. Don't reorder without care.
Susceptibility_all = xr.concat(all_stations, dim='station')
CCN_all = xr.concat(all_stationsCCN, dim='station')

In [ ]:
# Force evaluation of the lazy dask graph -> in-memory arrays for plotting.
Susceptibility_all = Susceptibility_all.compute()
CCN_all = CCN_all.compute()

## 5 · Diagnostic plots

Three views: (a) susceptibility vs height for selected cutoff radii at one station, (b) all-level susceptibility vs cutoff radius for every station and estimator, and (c) per-station CCN–CDNC hexbin with the four precomputed fits overlaid.

### 5a · Susceptibility vs height (single station, all estimators)

In [ ]:
def PLOTSusc_by_Level(ds = Susceptibility_all, Fit = 'OLS', station = 'SMR-II'):
    radii_to_plot = [20, 30, 40, 50]
    for r in radii_to_plot:
        ds[f'{Fit} slope'].sel(radius=r, station = station).plot(
            y='lev', 
            yincrease=False, 
            ylim=[1000, 600],
            xlim = [0,1],
            label=f'radius={r} nm'
        )
    
    plt.legend()
    plt.xlabel("Susceptibility")
    plt.ylabel("Pressure Level [hPa]")
    plt.title(f"Susceptibility ({Fit}) with height for selected cutoff radii")
    plt.show()

FitType = ['OLS', 'ODR', 'Deming', 'PCA']
for fit in FitType:
    print(fit)
    PLOTSusc_by_Level(Susceptibility_all, Fit = fit, station = 'SMR-II')

### 5b · All-level susceptibility vs cutoff radius (grid over stations)

One panel per station; markers are the four estimators. Empty panels are hidden.

In [ ]:
stations = Susceptibility_all.station.values
methods = ["OLS", "TLS", "Deming", "PCA"]

# Define number of columns and rows dynamically
ncols = 4
nrows = int(np.ceil(len(stations) / ncols))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(5 * ncols, 4 * nrows),
    sharey=True
)

axes = axes.flatten()

for i, station in enumerate(stations):
    ax = axes[i]
    for method in methods:
        var_name = f"All_Level_{method}_slope"
        if var_name in Susceptibility_all:
            Susceptibility_all[var_name].sel(station=station).plot(
                ax=ax, marker='x', label=method
            )
    ax.set_title(f"{station}", fontsize=11)
    ax.set_xlabel("Cutoff Radius (nm)")
    ax.set_ylabel("Susceptibility")
    ax.set_ylim(0, 2)
    ax.grid(True, alpha=0.3)

# Hide any unused subplots
for j in range(len(stations), len(axes)):
    axes[j].set_visible(False)

# Add a global legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Method", loc="lower right", fontsize=20)

fig.suptitle("All-level CCN–CDNC Susceptibility per Station", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


### 5c · CCN–CDNC fits per station (log–log hexbin)

Uses the precomputed slopes/intercepts (no refitting). `CCN_ds`/`Nor_ds` here are the last station's, carried over from section 4.

In [ ]:
def plot_station_fits_from_dataset(Susceptibility_all, station, CCN_ds, Nor_ds, radius=35, methods=None):
    """
    Plot Nd vs CCN (log–log) with precomputed fits from Susceptibility_all.
    No refitting is done here.
    """
    if methods is None:
        methods = ["OLS", "TLS", "Deming", "PCA"]

    # --- Prepare data ---
    x = CCN_all['CCN'].sel(radius=radius, station = station).to_numpy().ravel()
    y = CCN_all["CDNC"].sel(station = station).to_numpy().ravel()

    mask = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x, y = x[mask], y[mask]

    fits = []

    # --- Get slopes/intercepts from dataset ---
    for method in methods:
        slope_var = f"All_Level_{method}_slope"
        intercept_var = f"All_Level_{method}_intercept"

        if slope_var in Susceptibility_all and intercept_var in Susceptibility_all:
            slope = Susceptibility_all[slope_var].sel(station=station, radius=radius).compute().item()
            intercept = Susceptibility_all[intercept_var].sel(station=station, radius=radius).compute().item()

            fits.append({
                "slope": slope,
                "intercept": intercept,
                "label": f"{method}: slope={slope:.2f}",
                "style": {
                    "OLS": "r-",
                    "TLS": "g--",
                    "Deming": "b-.",
                    "PCA": "m:",
                }.get(method, "k-")
            })

    # --- Plot all fits on a single hexbin plot ---
    fig, ax = Function.plot_hexbin_regression_multi(
        x, y,
        fits=fits,
        lims=(1, 1e4),
        title=f"{station}: CCN–CDNC Fits (radius={radius} nm)"
    )

    plt.tight_layout()
    return fig, ax

In [ ]:
for station in stations:
    plot_station_fits_from_dataset(
        Susceptibility_all,
        station=station,
        CCN_ds=CCN_ds,
        Nor_ds=Nor_ds,
        radius=35
    )

### Inspect the assembled CCN dataset

In [ ]:
CCN_all